In [2]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# ============================================================
# 0. 데이터 불러오기
# ============================================================

train_full = pd.read_parquet(PROJECT_ROOT / 'data' / '2_train_full_fe.parquet')
test = pd.read_parquet(PROJECT_ROOT / 'data' / '2_test_fe.parquet')

In [15]:
# ============================================================
# 1. 컬럼군 정의
# ============================================================

# 종속변수 컬럼
target_col = 'target_maxrise_6h'  # 모델링 학습부터는 반드시 빼내야 함

# 시계열 컬럼군
time_cols = [
    'wl_t',          # 수위데이터 대표
    'sfc_rain_1h',   # 육상 누적강수데이터 대표
    'imerg_rain_1h', # 위성 누적강수데이터 대표
    'radar_reflectivity_mean', # 레이더반사도 대표
    'radar_reflectivity_p95', # 레이더반사도 대표
]

# 수위 컬럼군
wl_cols = [
    'wl_t',
    'wl_t_minus_1h',
    'wl_t_minus_3h',
    'wl_t_minus_6h',
    'wl_t_minus_12h',
    'wl_t_minus_24h',
]

# 수위차분(wl_diff) 컬럼군
wl_diff_cols = [
    'wl_t',
    'wl_diff_1h',
    'wl_diff_3h',
    'wl_diff_6h',
    'wl_diff_12h',
    'wl_diff_24h'
]


# 강수 컬럼군
rain_cols = [
    # IMERG강우
    'imerg_rain_1h',
    'imerg_rain_6h',
    'imerg_rain_24h',
    # 지상강우
    'sfc_rain_1h',
    'sfc_rain_6h',
    'sfc_rain_24h',
    # 레이더 반사도
    'radar_reflectivity_mean',
    'radar_reflectivity_p95',
    'radar_reflectivity_ge20_fraction'
]

# 강우강도 컬럼군
rain_int_cols = [
    'imerg_rain_1h',
    'imerg_int_1to6h',
    'imerg_int_6to24h',
    'sfc_rain_1h',
    'sfc_int_1to6h',
    'sfc_int_6to24h'
]

# 환경 컬럼군
env_cols = [
    'cat_area',       # km²
    'wsarea',         # km²
    'mean_elevation', # m
    'mean_slope',     # °
    'urban_ratio',    # %
    'forest_ratio',   # %
    'stream_order'
]

num_cols = wl_cols + rain_cols + env_cols
num_cols_fe_wl = num_cols + [
    col for col in wl_diff_cols if col != 'wl_t'
]
num_cols_fe = num_cols_fe_wl + [
    col for col in rain_int_cols if col not in ['imerg_rain_1h', 'sfc_rain_1h']
]

print(f"""<변수개수 검증>
    수위변수: {len(wl_cols)}
    강우변수: {len(rain_cols)}, 
    환경변수: {len(env_cols)}, 
    수위차분(wl_t포함): {len(wl_diff_cols)},
    강우강도(rain_1h포함): {len(rain_int_cols)}
    기존 숫자열(수위 + 강우 + 환경): {len(num_cols)}, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): {len(num_cols_fe_wl)},
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): {len(num_cols_fe)}
""")

<변수개수 검증>
    수위변수: 6
    강우변수: 9, 
    환경변수: 7, 
    수위차분(wl_t포함): 6,
    강우강도(rain_1h포함): 6
    기존 숫자열(수위 + 강우 + 환경): 22, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): 27,
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): 31



In [25]:
# 이상치 2: Train_full 수위 관측값 이상치 확인(상관계수분석에서 찾음)

cond_wl_out_station_vt1 = train_full['station_id'] == 'ST0445'
cond_wl_out_station_vt2 = train_full['station_id'] == 'ST0066'
cond_wl_out_ST0445_vt = (train_full[wl_cols] > 15).any(axis=1)
cond_wl_out_ST0066_vt = (train_full[wl_cols] > 15).any(axis=1)

# print(
#     f"Train의 wl_t 이상치 개수: "
#     f"{((cond_wl_out_station_vt1 & cond_wl_out_ST0445_vt) | (cond_wl_out_station_vt2 & cond_wl_out_ST0066_vt)).sum()}"
# )

# print("\nST0445 이상치 데이터")
# display(train_full.loc[cond_wl_out_station_vt1 & cond_wl_out_ST0445_vt])

print("\nST0066 이상치 데이터")
display(train_full.loc[cond_wl_out_station_vt2 & cond_wl_out_ST0066_vt])

# 시간대별 시계열 확인
cond_time_ST0445_error = (
    (
        train_full['timestamp'].between('2100-08-09 00:00:00', '2100-08-10 00:00:00') |
        train_full['timestamp'].between('2101-11-02 00:00:00', '2101-11-03 00:00:00') |
        train_full['timestamp'].between('2102-09-18 00:00:00', '2102-09-19 00:00:00')
    )
)

cond_time_ST0066_error = (
    (
        train_full['timestamp'].between('2100-08-18 00:00:00', '2100-08-20 00:00:00') |
        train_full['timestamp'].between('2102-03-31 00:00:00', '2102-04-02 00:00:00')
    )
)

print("\nST0445 시계열")
display(train_full.loc[
    cond_wl_out_station_vt1 & cond_time_ST0445_error,
    ['timestamp'] + wl_cols + [target_col]
].sort_values('timestamp'))

print("\nST0066 시계열")
display(train_full.loc[
    cond_wl_out_station_vt2 & cond_time_ST0066_error,
    ['timestamp'] + wl_cols + [target_col]
].sort_values('timestamp'))

print("\n이상치 인덱스")
print(train_full.index[
    (cond_wl_out_station_vt1 & cond_wl_out_ST0445_vt) |
    (cond_wl_out_station_vt2 & cond_wl_out_ST0066_vt)
].tolist())


ST0066 이상치 데이터


,timestamp,station_id,wl_t,wl_t_minus_1h,wl_t_minus_3h,wl_t_minus_6h,wl_t_minus_12h,wl_t_minus_24h,imerg_rain_1h,imerg_rain_6h,imerg_rain_24h,sfc_rain_1h,sfc_rain_6h,sfc_rain_24h,radar_reflectivity_mean,radar_reflectivity_p95,radar_reflectivity_ge20_fraction,cat_area,wsarea,mean_elevation,mean_slope,urban_ratio,forest_ratio,stream_order,target_maxrise_6h,wl_diff_1h,wl_diff_3h,wl_diff_6h,wl_diff_12h,wl_diff_24h,imerg_int_1to6h,imerg_int_6to24h,sfc_int_1to6h,sfc_int_6to24h
row_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
TR_000480207,2100-08-19 00:00:00,ST0066,23.56,23.53,23.54,23.55,23.54,23.56,0.281,1.338,2.874,0.0,0.0,0.0,12.588,15.310,0.0000,2.8934,26.0732,537.46,20.5159,1.7481,94.0463,1.7273,0.00,0.03,0.02,0.01,0.02,0.00,0.2114,0.085333,0.0,0.000000
TR_000612212,2100-08-19 01:00:00,ST0066,23.56,23.56,23.53,23.54,23.54,23.56,0.369,1.642,3.243,0.0,0.0,0.0,15.625,17.820,0.0000,2.8934,26.0732,537.46,20.5159,1.7481,94.0463,1.7273,0.00,0.00,0.03,0.02,0.02,0.00,0.2546,0.088944,0.0,0.000000
TR_000453262,2100-08-19 02:00:00,ST0066,23.56,23.56,23.53,23.54,23.54,23.55,0.420,2.062,3.663,0.0,0.0,0.0,11.627,14.100,0.0000,2.8934,26.0732,537.46,20.5159,1.7481,94.0463,1.7273,0.00,0.00,0.03,0.02,0.02,0.01,0.3284,0.088944,0.0,0.000000
TR_000575758,2100-08-19 03:00:00,ST0066,23.56,23.56,23.56,23.54,23.54,23.55,0.350,2.412,4.013,0.0,0.0,0.0,12.168,14.530,0.0000,2.8934,26.0732,537.46,20.5159,1.7481,94.0463,1.7273,0.00,0.00,0.00,0.02,0.02,0.01,0.4124,0.088944,0.0,0.000000
TR_000109053,2100-08-19 04:00:00,ST0066,23.56,23.56,23.56,23.53,23.55,23.55,0.645,2.855,4.658,0.0,0.0,0.0,2.859,10.849,0.0000,2.8934,26.0732,537.46,20.5159,1.7481,94.0463,1.7273,0.00,0.00,0.00,0.03,0.01,0.01,0.4420,0.100167,0.0,0.000000
TR_000598901,2100-08-19 05:00:00,ST0066,23.56,23.56,23.56,23.53,23.55,23.55,0.514,2.578,5.171,0.0,0.0,0.0,14.495,16.390,0.0000,2.8934,26.0732,537.46,20.5159,1.7481,94.0463,1.7273,0.00,0.00,0.00,0.03,0.01,0.01,0.4128,0.144056,0.0,0.000000
TR_000536275,2100-08-19 06:00:00,ST0066,23.56,23.56,23.56,23.56,23.55,23.54,1.085,3.382,6.256,0.0,0.0,0.0,10.638,13.920,0.0000,2.8934,26.0732,537.46,20.5159,1.7481,94.0463,1.7273,0.00,0.00,0.00,0.00,0.01,0.02,0.4594,0.159667,0.0,0.000000
TR_000292998,2100-08-19 07:00:00,ST0066,23.56,23.56,23.56,23.56,23.54,23.54,0.611,3.623,6.867,0.0,0.0,0.0,14.236,16.450,0.0000,2.8934,26.0732,537.46,20.5159,1.7481,94.0463,1.7273,0.00,0.00,0.00,0.00,0.02,0.02,0.6024,0.180222,0.0,0.000000
TR_000103144,2100-08-19 08:00:00,ST0066,23.56,23.56,23.56,23.56,23.54,23.54,0.555,3.759,7.422,0.0,0.0,0.0,13.759,18.511,0.0117,2.8934,26.0732,537.46,20.5159,1.7481,94.0463,1.7273,0.00,0.00,0.00,0.00,0.02,0.02,0.6408,0.203500,0.0,0.000000



ST0445 시계열


,timestamp,wl_t,wl_t_minus_1h,wl_t_minus_3h,wl_t_minus_6h,wl_t_minus_12h,wl_t_minus_24h,target_maxrise_6h
row_id,,,,,,,,
TR_000386997,2100-08-09 15:00:00,2.61,2.61,17.63,17.63,17.63,17.63,0.00
TR_000202940,2101-11-02 07:00:00,2.58,2.58,2.58,2.59,17.63,17.63,0.01
TR_000205403,2101-11-02 09:00:00,2.58,2.58,2.58,2.59,17.63,17.63,0.01
TR_000639479,2102-09-18 01:00:00,17.63,17.63,17.63,17.63,17.63,17.63,0.00
TR_000561875,2102-09-18 03:00:00,17.63,17.63,17.63,17.63,17.63,17.63,0.00
TR_000258437,2102-09-18 04:00:00,17.63,17.63,17.63,17.63,17.63,17.63,0.00
TR_000589829,2102-09-18 06:00:00,2.61,17.63,17.63,17.63,17.63,17.63,0.00
TR_000202521,2102-09-18 07:00:00,2.61,2.61,17.63,17.63,17.63,17.63,0.00
TR_000652873,2102-09-18 08:00:00,2.61,2.61,17.63,17.63,17.63,17.63,0.00



ST0066 시계열


,timestamp,wl_t,wl_t_minus_1h,wl_t_minus_3h,wl_t_minus_6h,wl_t_minus_12h,wl_t_minus_24h,target_maxrise_6h
row_id,,,,,,,,
TR_000480207,2100-08-19 00:00:00,23.56,23.53,23.54,23.55,23.54,23.56,0.00
TR_000612212,2100-08-19 01:00:00,23.56,23.56,23.53,23.54,23.54,23.56,0.00
TR_000453262,2100-08-19 02:00:00,23.56,23.56,23.53,23.54,23.54,23.55,0.00
TR_000575758,2100-08-19 03:00:00,23.56,23.56,23.56,23.54,23.54,23.55,0.00
TR_000109053,2100-08-19 04:00:00,23.56,23.56,23.56,23.53,23.55,23.55,0.00
TR_000598901,2100-08-19 05:00:00,23.56,23.56,23.56,23.53,23.55,23.55,0.00
TR_000536275,2100-08-19 06:00:00,23.56,23.56,23.56,23.56,23.55,23.54,0.00
TR_000292998,2100-08-19 07:00:00,23.56,23.56,23.56,23.56,23.54,23.54,0.00
TR_000103144,2100-08-19 08:00:00,23.56,23.56,23.56,23.56,23.54,23.54,0.00



이상치 인덱스
['TR_000386997', 'TR_000480207', 'TR_000612212', 'TR_000453262', 'TR_000575758', 'TR_000109053', 'TR_000598901', 'TR_000536275', 'TR_000292998', 'TR_000103144', 'TR_000157390', 'TR_000622988', 'TR_000508034', 'TR_000426744', 'TR_000165447', 'TR_000386146', 'TR_000616893', 'TR_000463562', 'TR_000677096', 'TR_000446779', 'TR_000202940', 'TR_000205403', 'TR_000437549', 'TR_000639479', 'TR_000561875', 'TR_000258437', 'TR_000589829', 'TR_000202521', 'TR_000652873', 'TR_000182368', 'TR_000333157', 'TR_000682989']


In [26]:
# 이상치 2: Test 수위 관측값 이상치 확인(상관계수분석에서 찾음)

cond_wl_out_station_vt1 = test['station_id'] == 'ST0445'
cond_wl_out_station_vt2 = test['station_id'] == 'ST0066'
cond_wl_out_ST0445_vt = (test[wl_cols] > 15).any(axis=1)
cond_wl_out_ST0066_vt = (test[wl_cols] > 15).any(axis=1)

print(
    f"Test의 wl_t 이상치 개수: "
    f"{((cond_wl_out_station_vt1 & cond_wl_out_ST0445_vt) | (cond_wl_out_station_vt2 & cond_wl_out_ST0066_vt)).sum()}"
)

# print("\nST0445 이상치 데이터")
# display(test.loc[cond_wl_out_station_vt1 & cond_wl_out_ST0445_vt])

# print("\nST0066 이상치 데이터")
# display(test.loc[cond_wl_out_station_vt2 & cond_wl_out_ST0066_vt])

# 시간대별 시계열 확인
cond_time_ST0445_error = (
    test['timestamp'].between('2103-04-05 00:00:00', '2103-04-07 00:00:00')
)

cond_time_ST0066_error = (
    test['timestamp'].between('2102-02-19 00:00:00', '2102-02-21 00:00:00')
)

print("\nST0445 시계열")
display(test.loc[
    cond_wl_out_station_vt1 & cond_time_ST0445_error,
    ['timestamp'] + wl_cols
].sort_values('timestamp'))

print("\nST0066 시계열")
display(test.loc[
    cond_wl_out_station_vt2 & cond_time_ST0066_error,
    ['timestamp'] + wl_cols
].sort_values('timestamp'))

# 최종 Test 예측값 사후보정 대상 row_id 저장
test_wl_outlier_ids = test.index[
    (cond_wl_out_station_vt1 & cond_wl_out_ST0445_vt) |
    (cond_wl_out_station_vt2 & cond_wl_out_ST0066_vt)
].tolist()

print("\n사후보정 대상 row_id")
print(test_wl_outlier_ids)


# 최종 test 예측셋에서 이상치 row_id의 예측값을 확인하여 0이 아닐경우 조치사항

# ② Test 데이터 분석 결과
# ST0445: 2103-04-06, 수위 17.63m가 일정하게 유지되는 패턴 확인
# ST0066: 2102-02-20, 수위 23.55~23.57m 범위에서 일정하게 유지되는 패턴 확인
# 두 관측소 모두 Train에서 확인한 수위 유지형 과대계측 패턴과 유사함

# ① Train 데이터 분석 결과
# ST0445: 17.63m의 과대계측 수위가 일정하게 유지되는 구간에서 종속변수 target_maxrise_6h가 0.00으로 확인됨
# ST0066: 23.53~23.56m의 과대계측 수위가 일정하게 유지되는 구간에서 종속변수가 대부분 0.00으로 확인됨
# ST0066의 2102-04-01 사례는 24시간 전 수위에 이상치가 잔존한 경우로, 이번 Test 이상치 패턴과 상이

# 사후보정
# 해당 이상치는 실제 수위 상승이 아니라 계측 이상으로 판단되는 비정상적인 고수위 유지 패턴임
# Train의 유사 사례에서 향후 6시간 최대수위상승량이 대부분 0에 가까운 것으로 확인됨
# 따라서 Test의 동일한 이상치 패턴에서도 향후 6시간의 유의미한 수위 상승이 발생하지 않는다고 가정하여 예측값을 0으로 보정

Test의 wl_t 이상치 개수: 14

ST0445 시계열


,timestamp,wl_t,wl_t_minus_1h,wl_t_minus_3h,wl_t_minus_6h,wl_t_minus_12h,wl_t_minus_24h
row_id,,,,,,,
TE_000091247,2103-04-06 00:00:00,17.63,17.63,17.63,17.63,17.63,17.63
TE_000097794,2103-04-06 01:00:00,17.63,17.63,17.63,17.63,17.63,17.63
TE_000173551,2103-04-06 02:00:00,17.63,17.63,17.63,17.63,17.63,17.63
TE_000042496,2103-04-06 03:00:00,17.63,17.63,17.63,17.63,17.63,17.63
TE_000125414,2103-04-06 04:00:00,17.63,17.63,17.63,17.63,17.63,17.63



ST0066 시계열


,timestamp,wl_t,wl_t_minus_1h,wl_t_minus_3h,wl_t_minus_6h,wl_t_minus_12h,wl_t_minus_24h
row_id,,,,,,,
TE_000138717,2102-02-20 00:00:00,23.56,23.56,23.57,23.56,23.56,23.57
TE_000155915,2102-02-20 01:00:00,23.56,23.56,23.56,23.56,23.56,23.57
TE_000085014,2102-02-20 02:00:00,23.56,23.56,23.56,23.56,23.56,23.56
TE_000143391,2102-02-20 03:00:00,23.56,23.56,23.56,23.57,23.55,23.56
TE_000173722,2102-02-20 04:00:00,23.55,23.56,23.56,23.56,23.55,23.56
TE_000152809,2102-02-20 05:00:00,23.56,23.55,23.56,23.56,23.56,23.56
TE_000089723,2102-02-20 06:00:00,23.55,23.56,23.56,23.56,23.56,23.56
TE_000112084,2102-02-20 07:00:00,23.55,23.55,23.55,23.56,23.56,23.56
TE_000182626,2102-02-20 08:00:00,23.55,23.55,23.56,23.56,23.56,23.56



사후보정 대상 row_id
['TE_000138717', 'TE_000155915', 'TE_000085014', 'TE_000143391', 'TE_000173722', 'TE_000152809', 'TE_000089723', 'TE_000112084', 'TE_000182626', 'TE_000091247', 'TE_000097794', 'TE_000173551', 'TE_000042496', 'TE_000125414']
